In [ ]:
# Instalacion de dependencias
!pip install -q matplotlib numpy pandas geopandas mapclassify

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import json
from urllib.request import urlopen
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

np.set_printoptions(precision=6, suppress=True)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 60)

In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def ahp_weights(matrix):
    """Calcula pesos AHP (vector propio + CI/CR)."""
    n = matrix.shape[0]
    RI = {1:0, 2:0, 3:0.58, 4:0.90, 5:1.12, 6:1.24, 7:1.32, 8:1.41, 9:1.45, 10:1.49}
    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    max_idx = np.argmax(eigenvalues.real)
    lambda_max = eigenvalues[max_idx].real
    w = eigenvectors[:, max_idx].real
    w = w / w.sum()
    CI = (lambda_max - n) / (n - 1)
    CR = CI / RI[n] if RI[n] > 0 else 0
    return w, lambda_max, CI, CR


def local_priority(value, rp_values, rp_priorities):
    """Interpola linealmente la prioridad local con truncamiento."""
    if value <= rp_values[0]:
        return rp_priorities[0]
    if value >= rp_values[-1]:
        return rp_priorities[-1]
    for i in range(len(rp_values) - 1):
        if rp_values[i] <= value <= rp_values[i + 1]:
            t = (value - rp_values[i]) / (rp_values[i + 1] - rp_values[i])
            return rp_priorities[i] + t * (rp_priorities[i + 1] - rp_priorities[i])
    return rp_priorities[-1]


def ahpsort_classify(alternatives_df, criteria_cols, weights,
                      rp_values_dict, rp_priorities_dict,
                      cp_values_dict, cp_priorities_global):
    """Clasifica alternativas usando AHPSort II."""
    results = []
    class_names = ['Alto', 'Medio', 'Bajo']
    for idx, row in alternatives_df.iterrows():
        global_priority = 0
        for j, crit in enumerate(criteria_cols):
            lp = local_priority(row[crit], rp_values_dict[crit], rp_priorities_dict[crit])
            global_priority += weights[j] * lp
        distances = [abs(global_priority - cp_priorities_global[k]) for k in range(3)]
        clase_idx = np.argmin(distances)
        results.append({
            'Provincia': row.iloc[0],
            'Prioridad_Global': global_priority,
            'Dist_Alto': distances[0],
            'Dist_Medio': distances[1],
            'Dist_Bajo': distances[2],
            'Clase': class_names[clase_idx]
        })
    return pd.DataFrame(results)


def calc_cp_globals(criteria_names, weights, cp_dict, rp_dict, rp_prio_dict):
    """Calcula prioridades locales y globales de los CPs."""
    cp_local = {}
    for crit in criteria_names:
        cp_local[crit] = [local_priority(v, rp_dict[crit], rp_prio_dict[crit]) for v in cp_dict[crit]]
    cp_global = []
    for i in range(3):
        gp = sum(weights[j] * cp_local[crit][i] for j, crit in enumerate(criteria_names))
        cp_global.append(gp)
    return cp_local, cp_global


def plot_local_priority_functions(rp_vals, rp_prios, cp_vals, cp_lp, criteria_names, title_prefix=""):
    """Dibuja funciones de prioridad local."""
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(f'{title_prefix}Funciones de Prioridad Local', fontsize=14, y=1.02)
    for j, crit in enumerate(criteria_names):
        ax = axes[j]
        ax.plot(rp_vals[crit], rp_prios[crit], 'b-o', markersize=5, label='RPs')
        for k, label in enumerate(['CP1', 'CP2', 'CP3']):
            ax.plot(cp_vals[crit][k], cp_lp[crit][k], 'ro', markersize=8)
            ax.annotate(label, (cp_vals[crit][k], cp_lp[crit][k]),
                       textcoords="offset points", xytext=(5, 5), fontsize=8, color='red')
        ax.set_title(crit, fontsize=11)
        ax.set_xlabel('Valor')
        ax.set_ylabel('Prioridad')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# DATOS COMUNES: Matriz de comparacion y pesos AHP
# ============================================================

comparison_matrix = np.array([
    [1,     5,   3,   7,   3],
    [1/5,   1,   3,   0.5, 1/4],
    [1/3, 1/3,   1,   1/5, 1/5],
    [1/7,   2,   5,   1,   1/5],
    [1/3,   4,   5,   5,   1]
])

criteria_names = ['Crimenes', 'Robos', 'Vehiculos', 'Orden_publico', 'Drogas']

weights, lambda_max, CI, CR = ahp_weights(comparison_matrix)

print("MATRIZ DE COMPARACION PAREADA")
print(pd.DataFrame(comparison_matrix, index=criteria_names, columns=criteria_names).round(3))
print(f"\nPesos: {dict(zip(criteria_names, weights.round(4)))}")
print(f"lambda_max = {lambda_max:.4f}")
print(f"CI = {CI:.4f}")
print(f"CR = {CR:.4f}  {'CONSISTENTE' if CR < 0.1 else 'NO CONSISTENTE'}")

In [ ]:
# ============================================================
# EJERCICIO 1: Provincias andaluzas - datos absolutos 2017
# ============================================================

andalucia_abs = pd.DataFrame({
    'Provincia': ['Almeria', 'Cadiz', 'Cordoba', 'Granada', 'Huelva', 'Jaen', 'Malaga', 'Sevilla'],
    'Crimenes': [23, 61, 20, 32, 8, 17, 58, 86],
    'Robos': [17284, 26571, 14508, 19139, 11222, 7548, 46137, 52849],
    'Vehiculos': [2101, 3945, 1691, 2299, 1409, 718, 6377, 9662],
    'Orden_publico': [387, 722, 219, 368, 156, 175, 830, 716],
    'Drogas': [267, 818, 148, 622, 139, 129, 934, 415]
})

cp_abs = {
    'Crimenes':      [20, 100, 160],
    'Robos':         [5000, 95000, 180000],
    'Vehiculos':     [5000, 19000, 27000],
    'Orden_publico': [300, 1800, 2500],
    'Drogas':        [300, 600, 900]
}

rp_abs = {
    'Crimenes':      [0, 41.2, 82.4, 123.6, 164.8, 206],
    'Robos':         [0, 50914.4, 101828.8, 152743.2, 203657.6, 254572],
    'Vehiculos':     [0, 7025.4, 14050.8, 21076.2, 28101.6, 35127],
    'Orden_publico': [0, 764.8, 1529.6, 2294.4, 3059.2, 3824],
    'Drogas':        [0, 321.6, 643.2, 964.8, 1286.4, 1608]
}

rp_priorities_abs = {
    'Crimenes':      [0.336, 0.132, 0.086, 0.032, 0.026, 0.023],
    'Robos':         [0.320, 0.139, 0.074, 0.047, 0.031, 0.022],
    'Vehiculos':     [0.459, 0.124, 0.069, 0.048, 0.034, 0.026],
    'Orden_publico': [0.308, 0.149, 0.071, 0.060, 0.032, 0.023],
    'Drogas':        [0.385, 0.112, 0.062, 0.033, 0.023, 0.018]
}

print("EJERCICIO 1 - Datos de criminalidad Andalucia 2017 (absolutos)")
print(andalucia_abs.to_string(index=False))

print("\nPerfiles Centrales (absolutos)")
print(pd.DataFrame(cp_abs, index=['CP1 Alto', 'CP2 Medio', 'CP3 Bajo']))

print("\nPuntos de Referencia")
print(pd.DataFrame(rp_abs, index=['RP1','RP2','RP3','RP4','RP5','RP6']))

print("\nPrioridades Locales de los RP")
print(pd.DataFrame(rp_priorities_abs, index=['RP1','RP2','RP3','RP4','RP5','RP6']))

In [ ]:
# Calculo de prioridades de CPs y clasificacion - Ejercicio 1
cp_local_abs, cp_global_abs = calc_cp_globals(criteria_names, weights, cp_abs, rp_abs, rp_priorities_abs)

print("Prioridades locales de los CPs:")
print(pd.DataFrame(cp_local_abs, index=['CP1 Alto', 'CP2 Medio', 'CP3 Bajo']).round(4))
print(f"\nPrioridades globales -> Alto={cp_global_abs[0]:.4f}, Medio={cp_global_abs[1]:.4f}, Bajo={cp_global_abs[2]:.4f}")

plot_local_priority_functions(rp_abs, rp_priorities_abs, cp_abs, cp_local_abs, criteria_names, "Ej.1 - ")

result_ej1 = ahpsort_classify(andalucia_abs, criteria_names, weights, rp_abs, rp_priorities_abs, cp_abs, cp_global_abs)
print("\nCLASIFICACION EJERCICIO 1")
print(result_ej1.sort_values('Prioridad_Global', ascending=False).to_string(index=False))

In [ ]:
# ============================================================
# EJERCICIO 2: Provincias andaluzas - por 10.000 hab.
# ============================================================

poblacion_and_2017 = {
    'Almeria': 707407, 'Cadiz': 1240284, 'Cordoba': 788287,
    'Granada': 914678, 'Huelva': 519229, 'Jaen': 643820,
    'Malaga': 1630615, 'Sevilla': 1939775
}

andalucia_norm = andalucia_abs.copy()
for crit in criteria_names:
    andalucia_norm[crit] = andalucia_norm.apply(
        lambda r: r[crit] / poblacion_and_2017[r['Provincia']] * 10000, axis=1
    )

print("EJERCICIO 2 - Datos normalizados por 10.000 hab.")
print(andalucia_norm.round(4).to_string(index=False))

cp_and_norm = {
    'Crimenes':      [0.26, 0.34, 0.41],
    'Robos':         [194.29, 215.32, 261.32],
    'Vehiculos':     [22.95, 28.44, 36.20],
    'Orden_publico': [2.87, 3.86, 5.32],
    'Drogas':        [2.06, 3.23, 6.25]
}

rp_and_norm = {
    'Crimenes':      [0.00, 0.10, 0.20, 0.30, 0.39, 0.49],
    'Robos':         [0.00, 56.59, 113.18, 169.77, 226.35, 282.94],
    'Vehiculos':     [0.00, 9.96, 19.93, 29.89, 39.85, 49.82],
    'Orden_publico': [0.00, 1.17, 2.33, 3.50, 4.66, 5.83],
    'Drogas':        [0.00, 1.36, 2.73, 4.09, 5.45, 6.81]
}

rp_priorities_and_norm = {
    'Crimenes':      [0.34, 0.13, 0.09, 0.03, 0.03, 0.02],
    'Robos':         [0.32, 0.14, 0.07, 0.05, 0.03, 0.02],
    'Vehiculos':     [0.46, 0.12, 0.07, 0.05, 0.04, 0.03],
    'Orden_publico': [0.31, 0.15, 0.07, 0.06, 0.03, 0.02],
    'Drogas':        [0.39, 0.11, 0.09, 0.05, 0.02, 0.02]
}

print("\nPerfiles Centrales (por 10.000 hab.)")
print(pd.DataFrame(cp_and_norm, index=['CP1 Alta', 'CP2 Media', 'CP3 Baja']))

print("\nPuntos de Referencia")
print(pd.DataFrame(rp_and_norm, index=['RP1','RP2','RP3','RP4','RP5','RP6']))

print("\nPrioridades Locales de los RP")
print(pd.DataFrame(rp_priorities_and_norm, index=['RP1','RP2','RP3','RP4','RP5','RP6']))

In [ ]:
# Calculo de prioridades de CPs y clasificacion - Ejercicio 2
cp_local_and, cp_global_and = calc_cp_globals(criteria_names, weights, cp_and_norm, rp_and_norm, rp_priorities_and_norm)

print("Prioridades locales de los CPs:")
print(pd.DataFrame(cp_local_and, index=['CP1 Alta', 'CP2 Media', 'CP3 Baja']).round(4))
print(f"\nPrioridades globales -> Alta={cp_global_and[0]:.4f}, Media={cp_global_and[1]:.4f}, Baja={cp_global_and[2]:.4f}")

plot_local_priority_functions(rp_and_norm, rp_priorities_and_norm, cp_and_norm, cp_local_and, criteria_names, "Ej.2 - ")

result_ej2 = ahpsort_classify(andalucia_norm, criteria_names, weights, rp_and_norm, rp_priorities_and_norm, cp_and_norm, cp_global_and)
print("\nCLASIFICACION EJERCICIO 2")
print(result_ej2.sort_values('Prioridad_Global', ascending=False).to_string(index=False))

In [ ]:
# ============================================================
# EJERCICIO 3: Todas las provincias - 2017 por 10.000 hab.
# ============================================================

data_2017 = pd.DataFrame({
    'Provincia': [
        'Araba/Alava', 'Albacete', 'Alicante/Alacant', 'Almeria', 'Avila',
        'Badajoz', 'Balears (Illes)', 'Barcelona', 'Burgos', 'Caceres',
        'Cadiz', 'Castellon/Castello', 'Ciudad Real', 'Cordoba', 'Coruna (A)',
        'Cuenca', 'Girona', 'Granada', 'Guadalajara', 'Gipuzkoa',
        'Huelva', 'Huesca', 'Jaen', 'Leon', 'Lleida',
        'Rioja (La)', 'Lugo', 'Madrid', 'Malaga', 'Murcia',
        'Navarra', 'Ourense', 'Asturias', 'Palencia', 'Palmas (Las)',
        'Pontevedra', 'Salamanca', 'Santa Cruz de Tenerife', 'Cantabria', 'Segovia',
        'Sevilla', 'Soria', 'Tarragona', 'Teruel', 'Toledo',
        'Valencia/Valencia', 'Valladolid', 'Bizkaia', 'Zamora', 'Zaragoza'
    ],
    'Crimenes': [
        0.214347, 0.333306, 0.202703, 0.325469, 0.124456,
        0.294168, 0.322581, 0.369438, 0.027920, 0.299973,
        0.492160, 0.243279, 0.318359, 0.253737, 0.241008,
        0.050323, 0.317580, 0.350517, 0.118432, 0.250250,
        0.154163, 0.364130, 0.264187, 0.213531, 0.439424,
        0.190246, 0.089919, 0.252029, 0.355694, 0.401286,
        0.575218, 0.192505, 0.202906, 0.000000, 0.336217,
        0.180327, 0.239806, 0.387043, 0.172326, 0.129715,
        0.443407, 0.000000, 0.315779, 0.737670, 0.363985,
        0.377848, 0.307025, 0.182879, 0.225474, 0.136342
    ],
    'Robos': [
        209.569653, 183.267014, 322.001696, 244.583060, 129.558183,
        149.422549, 399.874910, 406.464304, 158.136756, 93.691568,
        214.379939, 254.730916, 132.775410, 184.060521, 151.977963,
        159.069636, 324.486977, 209.641838, 188.622636, 174.827119,
        216.252674, 111.560204, 117.298954, 167.814040, 207.963292,
        150.230990, 99.180539, 391.216846, 282.942325, 248.164797,
        197.750741, 112.615503, 119.067404, 103.555909, 215.142483,
        147.168174, 168.823422, 268.498404, 165.484797, 126.277694,
        272.483961, 127.667233, 289.885094, 76.422596, 235.396547,
        272.841378, 169.151651, 270.947887, 135.115330, 175.482388
    ],
    'Vehiculos': [
        2.020981, 12.306683, 35.193598, 29.730908, 7.654014,
        11.928505, 41.326202, 46.594024, 12.759269, 6.749393,
        31.829019, 21.738753, 11.819061, 21.453429, 16.986612,
        14.442577, 45.652070, 25.182433, 24.120643, 3.864965,
        27.152024, 7.009495, 11.158009, 10.697905, 26.111049,
        9.861089, 9.771186, 53.981876, 39.107944, 30.715384,
        12.950186, 7.283111, 8.493082, 3.917008, 25.298052,
        14.426172, 12.649766, 36.590413, 11.942202, 6.356042,
        49.816270, 7.423821, 40.861799, 2.876912, 24.983948,
        29.637420, 9.133997, 5.077062, 7.384275, 15.784186
    ],
    'Orden_publico': [
        4.164447, 4.512450, 5.735943, 5.476374, 3.173615,
        2.735761, 5.681009, 3.057727, 4.104185, 2.374786,
        5.825235, 4.083619, 2.964714, 2.778416, 2.526123,
        1.861935, 4.856322, 4.030942, 3.395050, 5.199630,
        3.006186, 3.186134, 2.719570, 2.839963, 2.682800,
        3.995168, 2.607648, 5.876582, 5.090104, 3.740802,
        4.570654, 2.727156, 3.951844, 3.794602, 4.225429,
        2.938272, 2.368084, 4.178075, 2.843381, 3.113163,
        3.691622, 1.912196, 4.433537, 2.729378, 2.926441,
        4.919890, 2.225932, 3.370194, 2.592952, 3.838546
    ],
    'Drogas': [
        1.439184, 2.051114, 3.429513, 3.778273, 2.115744,
        1.867966, 4.327961, 2.575306, 2.066052, 1.849834,
        6.599781, 2.519680, 1.910151, 1.877651, 0.999738,
        1.710967, 5.782596, 6.813168, 1.460661, 1.779552,
        2.678589, 1.957197, 2.004712, 0.854124, 2.058356,
        2.536614, 1.498648, 2.471115, 5.727900, 1.992827,
        2.083223, 1.283368, 1.256087, 1.530081, 3.116822,
        1.792664, 1.828521, 2.520739, 1.085655, 1.751154,
        2.139697, 1.237304, 3.877766, 0.958971, 2.606134,
        3.231384, 0.556483, 1.367236, 1.860161, 2.244396
    ]
})

print(f"Datos nacionales 2017 por 10.000 hab. - {len(data_2017)} provincias")
print(data_2017.to_string(index=False))

In [ ]:
# CPs, RPs y prioridades nacionales 2017

cp_nac_2017 = {
    'Crimenes':      [0.17, 0.25, 0.36],
    'Robos':         [129.18, 179.38, 268.99],
    'Vehiculos':     [7.42, 13.69, 30.94],
    'Orden_publico': [2.73, 3.38, 4.52],
    'Drogas':        [1.46, 2.00, 2.77]
}

rp_nac_2017 = {
    'Crimenes':      [0.00, 0.15, 0.30, 0.44, 0.59, 0.74],
    'Robos':         [0.00, 81.29, 162.59, 243.88, 325.17, 406.46],
    'Vehiculos':     [0.00, 10.80, 21.59, 32.39, 43.19, 53.98],
    'Orden_publico': [0.00, 1.18, 2.35, 3.53, 4.70, 5.88],
    'Drogas':        [0.00, 1.36, 2.73, 4.09, 5.45, 6.81]
}

rp_priorities_nac_2017 = {
    'Crimenes':      [0.336, 0.132, 0.086, 0.032, 0.026, 0.023],
    'Robos':         [0.320, 0.139, 0.074, 0.047, 0.031, 0.022],
    'Vehiculos':     [0.459, 0.124, 0.069, 0.048, 0.034, 0.026],
    'Orden_publico': [0.308, 0.149, 0.071, 0.060, 0.032, 0.023],
    'Drogas':        [0.385, 0.112, 0.062, 0.033, 0.023, 0.018]
}

cp_local_nac_2017, cp_global_nac_2017 = calc_cp_globals(
    criteria_names, weights, cp_nac_2017, rp_nac_2017, rp_priorities_nac_2017
)

print("Perfiles Centrales nacionales (por 10.000 hab.)")
print(pd.DataFrame(cp_nac_2017, index=['CP1 Alta', 'CP2 Media', 'CP3 Baja']))

print("\nPuntos de Referencia nacionales")
print(pd.DataFrame(rp_nac_2017, index=['RP1','RP2','RP3','RP4','RP5','RP6']))

print("\nPrioridades locales de los RP")
print(pd.DataFrame(rp_priorities_nac_2017, index=['RP1','RP2','RP3','RP4','RP5','RP6']))

print("\nPrioridades locales de los CPs:")
cp_df = pd.DataFrame(cp_local_nac_2017, index=['CP1 Alta', 'CP2 Media', 'CP3 Baja'])
cp_df['Global'] = cp_global_nac_2017
print(cp_df.round(4))

plot_local_priority_functions(rp_nac_2017, rp_priorities_nac_2017, cp_nac_2017, cp_local_nac_2017, criteria_names, "Ej.3 - Nacional 2017 - ")

In [ ]:
# Clasificacion nacional 2017

result_2017 = ahpsort_classify(
    data_2017, criteria_names, weights,
    rp_nac_2017, rp_priorities_nac_2017,
    cp_nac_2017, cp_global_nac_2017
)

print("CLASIFICACION NACIONAL 2017")
print(result_2017.sort_values('Prioridad_Global', ascending=False).to_string(index=False))

print("\n--- Resumen por clases 2017 ---")
for clase in ['Alto', 'Medio', 'Bajo']:
    provs = result_2017[result_2017['Clase'] == clase]['Provincia'].tolist()
    print(f"{clase} ({len(provs)}): {', '.join(provs)}")

In [ ]:
# ============================================================
# EJERCICIO 4: Todas las provincias - 2023 por 10.000 hab.
# ============================================================

data_2023 = pd.DataFrame({
    'Provincia': [
        'Albacete', 'Alicante/Alacant', 'Almeria', 'Araba/Alava', 'Asturias',
        'Avila', 'Badajoz', 'Balears (Illes)', 'Barcelona', 'Bizkaia',
        'Burgos', 'Caceres', 'Cadiz', 'Cantabria', 'Castellon/Castello',
        'Ciudad Real', 'Cordoba', 'Coruna (A)', 'Cuenca', 'Gipuzkoa',
        'Girona', 'Granada', 'Guadalajara', 'Huelva', 'Huesca',
        'Jaen', 'Leon', 'Lleida', 'Lugo', 'Madrid',
        'Malaga', 'Murcia', 'Navarra', 'Ourense', 'Palencia',
        'Palmas (Las)', 'Pontevedra', 'Rioja (La)', 'Salamanca', 'Santa Cruz de Tenerife',
        'Segovia', 'Sevilla', 'Soria', 'Tarragona', 'Teruel',
        'Toledo', 'Valencia/Valencia', 'Valladolid', 'Zamora', 'Zaragoza'
    ],
    'Crimenes': [
        0.309938, 0.394800, 0.490768, 0.207909, 0.208896,
        0.315643, 0.225534, 0.417620, 0.370338, 0.147275,
        0.111988, 0.154865, 0.503783, 0.169947, 0.380825,
        0.285059, 0.310079, 0.249238, 0.304354, 0.357130,
        0.333636, 0.419273, 0.254918, 0.207225, 0.308265,
        0.306332, 0.312875, 0.358108, 0.370421, 0.300295,
        0.405344, 0.515312, 0.565690, 0.098506, 0.253563,
        0.333258, 0.285943, 0.465131, 0.152826, 0.367304,
        0.064409, 0.505822, 0.447017, 0.613522, 0.073944,
        0.288265, 0.376465, 0.345442, 0.180325, 0.399012
    ],
    'Robos': [
        148.925289, 213.945447, 149.710845, 160.297725, 106.298425,
        91.915130, 116.631007, 261.714029, 360.466799, 219.621140,
        127.190772, 79.703898, 156.484524, 133.969161, 192.697433,
        100.788594, 138.101310, 132.318462, 115.350083, 162.521445,
        259.716830, 142.380891, 136.745351, 161.692011, 113.309582,
        98.896882, 119.451217, 176.793280, 87.851437, 264.561626,
        217.538251, 186.497919, 179.636351, 96.897061, 70.617171,
        158.437827, 107.789821, 138.919036, 131.582969, 168.489061,
        98.288011, 222.433975, 98.567310, 231.616181, 66.771668,
        137.269113, 255.393705, 115.396942, 91.845690, 141.199268
    ],
    'Vehiculos': [
        14.024702, 26.420804, 19.299130, 2.435504, 8.405593,
        6.754752, 11.742789, 41.987503, 50.042201, 3.751172,
        11.506803, 5.317035, 27.236256, 10.655672, 21.607677,
        8.449953, 20.426436, 16.992664, 14.000274, 3.653711,
        37.540191, 17.813737, 25.601060, 29.463626, 7.750675,
        9.689766, 12.313867, 19.158760, 8.149255, 42.306653,
        31.759534, 27.169835, 11.641305, 12.805779, 4.183782,
        18.750143, 13.428718, 10.170858, 17.391570, 21.492009,
        8.566386, 47.746537, 4.470173, 38.923222, 3.253547,
        20.247194, 41.870413, 9.039075, 6.070952, 11.683904
    ],
    'Orden_publico': [
        5.217292, 8.326681, 8.608340, 6.860992, 5.083146,
        3.345812, 4.435497, 7.634091, 5.007321, 5.648416,
        5.739403, 3.303789, 8.084514, 4.775509, 6.772061,
        5.273585, 4.031023, 3.685156, 4.108776, 7.170064,
        6.524431, 5.429051, 3.969439, 5.369011, 4.668020,
        4.401508, 3.888590, 4.610636, 2.901629, 8.317889,
        7.855675, 6.602437, 8.023866, 3.940240, 5.578376,
        5.191807, 3.812570, 4.620298, 3.362166, 4.774955,
        5.732394, 5.334124, 7.599294, 6.253200, 2.809882,
        5.902572, 7.480355, 3.454424, 3.666615, 8.420187
    ],
    'Drogas': [
        2.479505, 5.665629, 6.194291, 3.801762, 1.890015,
        2.462012, 3.428113, 6.481461, 5.539575, 1.862591,
        2.295761, 2.245544, 6.309279, 1.937395, 2.831351,
        2.484083, 1.963832, 1.397511, 2.485556, 3.488882,
        6.833353, 12.664202, 2.257846, 3.315600, 4.403793,
        2.595761, 1.340893, 3.021533, 1.913840, 4.940295,
        7.130623, 3.736013, 2.352080, 1.116401, 3.486485,
        3.543058, 2.488761, 2.883810, 2.047865, 3.390501,
        2.898401, 3.157556, 3.240875, 5.474500, 1.922551,
        3.651358, 4.856396, 1.132283, 3.366072, 4.358444
    ]
})

print(f"Datos nacionales 2023 por 10.000 hab. - {len(data_2023)} provincias")
print(data_2023.to_string(index=False))

In [ ]:
# Clasificacion 2023 con calibracion de 2017

result_2023 = ahpsort_classify(
    data_2023, criteria_names, weights,
    rp_nac_2017, rp_priorities_nac_2017,
    cp_nac_2017, cp_global_nac_2017
)

print("CLASIFICACION NACIONAL 2023 (calibracion 2017, con truncamiento)")
print(result_2023.sort_values('Prioridad_Global', ascending=False).to_string(index=False))

print("\n--- Resumen por clases 2023 ---")
for clase in ['Alto', 'Medio', 'Bajo']:
    provs = result_2023[result_2023['Clase'] == clase]['Provincia'].tolist()
    print(f"{clase} ({len(provs)}): {', '.join(provs)}")

In [ ]:
# ============================================================
# MAPAS
# ============================================================

url = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/spain-provinces.geojson"
try:
    with urlopen(url) as response:
        geojson_data = json.loads(response.read().decode())
    geo_df = gpd.GeoDataFrame.from_features(geojson_data['features'])
except:
    url2 = "https://raw.githubusercontent.com/martgnz/bcn-geodata/master/provincias/provincias.geojson"
    with urlopen(url2) as response:
        geojson_data = json.loads(response.read().decode())
    geo_df = gpd.GeoDataFrame.from_features(geojson_data['features'])

# Detectar columna de nombre
name_col = None
for col in ['name', 'NAME', 'nombre', 'NOMBRE', 'provincia', 'Texto']:
    if col in geo_df.columns:
        name_col = col
        break
if name_col is None:
    name_col = geo_df.columns[0]

print(f"GeoJSON: {len(geo_df)} poligonos, columna nombre: '{name_col}'")
print(f"Nombres: {sorted(geo_df[name_col].unique())}")

In [ ]:
# Mapeo de nombres GeoJSON -> datos

name_mapping = {
    'Araba/Alava': ['Alava', 'Araba', 'Araba/Alava', 'Araba/Álava', 'Álava'],
    'Albacete': ['Albacete'],
    'Alicante/Alacant': ['Alicante', 'Alacant', 'Alicante/Alacant'],
    'Almeria': ['Almería', 'Almeria'],
    'Avila': ['Ávila', 'Avila'],
    'Badajoz': ['Badajoz'],
    'Balears (Illes)': ['Illes Balears', 'Islas Baleares', 'Balears', 'Baleares'],
    'Barcelona': ['Barcelona'],
    'Burgos': ['Burgos'],
    'Caceres': ['Cáceres', 'Caceres'],
    'Cadiz': ['Cádiz', 'Cadiz'],
    'Castellon/Castello': ['Castellón', 'Castelló', 'Castellon'],
    'Ciudad Real': ['Ciudad Real'],
    'Cordoba': ['Córdoba', 'Cordoba'],
    'Coruna (A)': ['A Coruña', 'La Coruña', 'Coruña'],
    'Cuenca': ['Cuenca'],
    'Girona': ['Girona', 'Gerona'],
    'Granada': ['Granada'],
    'Guadalajara': ['Guadalajara'],
    'Gipuzkoa': ['Gipuzkoa', 'Guipúzcoa', 'Guipuzcoa'],
    'Huelva': ['Huelva'],
    'Huesca': ['Huesca'],
    'Jaen': ['Jaén', 'Jaen'],
    'Leon': ['León', 'Leon'],
    'Lleida': ['Lleida', 'Lérida', 'Lerida'],
    'Rioja (La)': ['La Rioja', 'Rioja'],
    'Lugo': ['Lugo'],
    'Madrid': ['Madrid'],
    'Malaga': ['Málaga', 'Malaga'],
    'Murcia': ['Murcia'],
    'Navarra': ['Navarra'],
    'Ourense': ['Ourense', 'Orense'],
    'Asturias': ['Asturias'],
    'Palencia': ['Palencia'],
    'Palmas (Las)': ['Las Palmas'],
    'Pontevedra': ['Pontevedra'],
    'Salamanca': ['Salamanca'],
    'Santa Cruz de Tenerife': ['Santa Cruz de Tenerife'],
    'Cantabria': ['Cantabria'],
    'Segovia': ['Segovia'],
    'Sevilla': ['Sevilla'],
    'Soria': ['Soria'],
    'Tarragona': ['Tarragona'],
    'Teruel': ['Teruel'],
    'Toledo': ['Toledo'],
    'Valencia/Valencia': ['Valencia', 'València', 'Valencia/València'],
    'Valladolid': ['Valladolid'],
    'Bizkaia': ['Bizkaia', 'Vizcaya'],
    'Zamora': ['Zamora'],
    'Zaragoza': ['Zaragoza']
}

reverse_mapping = {}
for key, aliases in name_mapping.items():
    for alias in aliases:
        reverse_mapping[alias.lower()] = key
    reverse_mapping[key.lower()] = key

def match_province(geo_name):
    if geo_name.lower() in reverse_mapping:
        return reverse_mapping[geo_name.lower()]
    for alias_lower, key in reverse_mapping.items():
        if alias_lower in geo_name.lower() or geo_name.lower() in alias_lower:
            return key
    return None

geo_df['Provincia_key'] = geo_df[name_col].apply(match_province)
matched = geo_df['Provincia_key'].notna().sum()
print(f"Provincias emparejadas: {matched}/{len(geo_df)}")
unmatched = geo_df[geo_df['Provincia_key'].isna()][name_col].tolist()
if unmatched:
    print(f"Sin emparejar: {unmatched}")

In [ ]:
# Funcion de dibujado de mapa

def plot_map(geo_df, result_df, title, ax=None):
    merged = geo_df.merge(result_df, left_on='Provincia_key', right_on='Provincia', how='left')
    color_map = {'Alto': '#2ecc71', 'Medio': '#f39c12', 'Bajo': '#e74c3c'}
    merged['color'] = merged['Clase'].map(color_map).fillna('#cccccc')
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    merged.plot(ax=ax, color=merged['color'], edgecolor='black', linewidth=0.5)
    legend_elements = [
        Patch(facecolor='#2ecc71', edgecolor='black', label='Alto (mas seguridad)'),
        Patch(facecolor='#f39c12', edgecolor='black', label='Medio'),
        Patch(facecolor='#e74c3c', edgecolor='black', label='Bajo (menos seguridad)'),
        Patch(facecolor='#cccccc', edgecolor='black', label='Sin datos')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_axis_off()
    return ax

In [ ]:
# Mapa 2017
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
plot_map(geo_df, result_2017, 'Clasificacion de Seguridad Provincial - Espana 2017\n(AHPSort II, por 10.000 hab.)', ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa 2023
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
plot_map(geo_df, result_2023, 'Clasificacion de Seguridad Provincial - Espana 2023\n(AHPSort II, calibracion 2017, por 10.000 hab.)', ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Mapas comparativos lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))
plot_map(geo_df, result_2017, 'Seguridad Provincial 2017', ax=ax1)
plot_map(geo_df, result_2023, 'Seguridad Provincial 2023', ax=ax2)
plt.suptitle('Comparacion de Clasificacion Provincial de Seguridad (AHPSort II)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TABLA DE TRANSICION Y MATRIZ DE CAMBIOS
# ============================================================

transition = result_2017[['Provincia', 'Clase']].merge(
    result_2023[['Provincia', 'Clase']],
    on='Provincia', suffixes=('_2017', '_2023')
)

clases_order = ['Alto', 'Medio', 'Bajo']
transition['Cambio'] = transition.apply(
    lambda r: 'Igual' if r['Clase_2017'] == r['Clase_2023'] else
    'Mejora' if clases_order.index(r['Clase_2023']) < clases_order.index(r['Clase_2017']) else
    'Empeora', axis=1
)

print("TABLA DE TRANSICION (2017 -> 2023)")
print(transition.to_string(index=False))

In [ ]:
# Matriz de cambios entre clases

confusion = pd.crosstab(
    transition['Clase_2017'], transition['Clase_2023'],
    rownames=['Clase 2017'], colnames=['Clase 2023']
)
for c in clases_order:
    if c not in confusion.columns:
        confusion[c] = 0
    if c not in confusion.index:
        confusion.loc[c] = 0
confusion = confusion.reindex(index=clases_order, columns=clases_order, fill_value=0)

print("MATRIZ DE CAMBIOS ENTRE CLASES")
print(confusion)

print(f"\nProvincias que mejoran: {(transition['Cambio'] == 'Mejora').sum()}")
print(f"Provincias sin cambio:  {(transition['Cambio'] == 'Igual').sum()}")
print(f"Provincias que empeoran: {(transition['Cambio'] == 'Empeora').sum()}")

In [ ]:
# Provincias que cambiaron de clase

cambios = transition[transition['Cambio'] != 'Igual'].sort_values('Cambio')
if len(cambios) > 0:
    print("PROVINCIAS QUE CAMBIARON DE CLASE")
    for _, row in cambios.iterrows():
        print(f"  {row['Provincia']:30s}: {row['Clase_2017']:6s} -> {row['Clase_2023']:6s}  ({row['Cambio']})")
else:
    print("Ninguna provincia cambio de clase.")